# HDFS Session Sequencer

Builds **per-block event sequences** from a parsed HDFS annotated log DataFrame.

**Input** — `*_hdfs_annotated.parquet` produced by `Parser.ipynb`  
**Output** — `*_hdfs_sequences.parquet` — flat table of all log lines with a `block_id` column, sorted by `(block_id, timestamp)`, ready for graph or model ingestion.

Each sequence is the ordered time-series of Drain template firings for one HDFS block — the unit of anomaly labelling in the benchmark.

In [ ]:
import os, sys, json
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

PROCESSED_DIR = Path("../data/processed")

## Step 1 — Load annotated log

Auto-picks the newest `*_hdfs_annotated.parquet` from `data/processed/`.  
Override `input_file` manually to pin a specific run.

In [ ]:
candidates = sorted(PROCESSED_DIR.glob("*_hdfs_annotated.parquet"))
if not candidates:
    raise FileNotFoundError(
        f"No *_hdfs_annotated.parquet found in {PROCESSED_DIR}.\n"
        "Run Parser.ipynb first to generate the annotated log file."
    )

input_file = candidates[-1]   # newest by YYYYMMDD_HHMM prefix
# input_file = PROCESSED_DIR / "20260407_1030_hdfs_annotated.parquet"  # pin a specific run

RUN_TAG = input_file.stem.replace("_hdfs_annotated", "")
print(f"Run tag    : {RUN_TAG}")
print(f"Input file : {input_file}")

df = pd.read_parquet(input_file, engine="fastparquet")
df["parameters"] = df["parameters"].apply(json.loads)

print(f"\nRows loaded : {len(df):,}")
print(f"Columns     : {df.columns.tolist()}")
df.head(3)

## Step 2 — Build per-block sequences

Drop lines with no `block_id` (NameNode bookkeeping lines that don't reference a specific block),  
then sort by `(block_id, timestamp)` so each group is a clean ordered time-series.

In [ ]:
import time

t0 = time.time()

# Filter without .copy() — avoids duplicating the full DataFrame in memory
df_blocks = df.loc[df["block_id"].notna()].sort_values(["block_id", "timestamp"])

# dict(groupby) is a single pass; no per-group reset_index allocation
# (86 k groups × reset_index = 86 k DataFrame copies — the old bottleneck)
sequences = dict(df_blocks.groupby("block_id", sort=False))

elapsed = time.time() - t0
n_dropped = len(df) - len(df_blocks)

print(f"Total log lines          : {len(df):,}")
print(f"Lines without block_id   : {n_dropped:,}  ({100 * n_dropped / len(df):.1f}%)")
print(f"Lines with block_id      : {len(df_blocks):,}")
print(f"Unique blocks            : {len(sequences):,}")
print(f"Built in                 : {elapsed:.2f}s")


## Step 3 — Sequence statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

seq_lengths = pd.Series({bid: len(seq) for bid, seq in sequences.items()}, name="length")

print("Sequence length distribution:")
print(seq_lengths.describe().round(2).to_string())

print("\nPercentiles:")
pcts = [50, 75, 90, 95, 99, 100]
vals = np.percentile(seq_lengths, pcts)
print("  " + "  ".join(f"p{p}={v:.0f}" for p, v in zip(pcts, vals)))

# ── Distribution plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"Sequence length distribution  [{RUN_TAG}]", fontsize=12)

for ax, log_y in zip(axes, [False, True]):
    ax.hist(seq_lengths, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
    ax.axvline(seq_lengths.median(), color="red",    linestyle="--", linewidth=1.2,
               label=f"median={seq_lengths.median():.0f}")
    ax.axvline(seq_lengths.mean(),   color="orange", linestyle=":",  linewidth=1.2,
               label=f"mean={seq_lengths.mean():.1f}")
    ax.set_xlabel("Events per block")
    ax.set_ylabel("Number of blocks")
    ax.set_title("Linear y" if not log_y else "Log y")
    if log_y:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4 — Inspect sample sequences

In [ ]:
def display_sequence(bid: str) -> None:
    """Pretty-print the ordered event sequence for a single block."""
    if bid not in sequences:
        print(f"Block '{bid}' not found.")
        return
    seq = sequences[bid]
    print("=" * 100)
    print(f"Block ID    : {bid}")
    print(f"Events      : {len(seq)}")
    print(f"Time span   : {seq['timestamp'].min()} → {seq['timestamp'].max()}")
    print("=" * 100)
    for i, row in seq.iterrows():
        print(f"  [{i:>3}] {row['timestamp']}  cid={row['cluster_id']:>3}  {row['template'][:80]}")
        if row["parameters"]:
            print(f"         params={row['parameters']}")
    print()


# Show the shortest, median, and longest sequences
sorted_bids = seq_lengths.sort_values().index.tolist()
sample_bids = {
    "shortest" : sorted_bids[0],
    "median"   : sorted_bids[len(sorted_bids) // 2],
    "longest"  : sorted_bids[-1],
}

for label, bid in sample_bids.items():
    print(f"\n{'━'*40} {label.upper()} {'━'*40}")
    display_sequence(bid)

## Step 5 — Save sequences to Parquet

In [ ]:
SEQUENCES_PATH = PROCESSED_DIR / f"{RUN_TAG}_hdfs_sequences.parquet"

df_save = df_blocks.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# Cast Arrow-backed string columns to plain numpy object (fastparquet requirement)
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

df_save.to_parquet(SEQUENCES_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows across {df_save['block_id'].nunique():,} blocks")
print(f"  → {SEQUENCES_PATH}  ({SEQUENCES_PATH.stat().st_size / 1_048_576:.1f} MB)")

# ── Reload in a later session ────────────────────────────────────────────────
# df_blocks = pd.read_parquet(SEQUENCES_PATH, engine="fastparquet")
# df_blocks["parameters"] = df_blocks["parameters"].apply(json.loads)
# sequences = {bid: grp.reset_index(drop=True) for bid, grp in df_blocks.groupby("block_id")}